Cleaning and Validating the Data

We did a little data cleaning previously, but now we are going to learn repeatable data-cleaning habits, as well as how to prepare the dataset for feature engineering. So, we're going to proceed as if we had no information from the previous notebook.

First, load the dataset.


In [30]:
import pandas as pd

df = pd.read_csv("../data/raw/books.csv")
print(f"Rows loaded: {len(df)}")
df.head()


Rows loaded: 100


,book_id,title,author,genres,publication_year,page_count,average_rating,rating_count,description
0,B001,Dune,Frank Herbert,Science Fiction;Politics;Adventure,1965,412,4.27,850000,A young nobleman becomes entangled in a strugg...
1,B002,Hyperion,Dan Simmons,Science Fiction;Space Opera;Adventure,1989,482,4.22,185000,Seven travelers journey toward a mysterious wo...
2,B003,The Left Hand of Darkness,Ursula K. Le Guin,Science Fiction;Politics;Literary,1969,304,4.08,190000,An envoy visits a distant planet and must navi...
3,B004,The Dispossessed,Ursula K. Le Guin,Science Fiction;Politics;Literary,1974,387,4.23,170000,A physicist moves between contrasting worlds w...
4,B005,Foundation,Isaac Asimov,Science Fiction;Space Opera;Politics,1951,296,4.17,500000,A mathematician predicts the collapse of a vas...


In [31]:
print(df.columns.tolist())

['book_id', 'title', 'author', 'genres', 'publication_year', 'page_count', 'average_rating', 'rating_count', 'description']


Check for any exact duplicates

In [32]:
print(df.duplicated().sum())

#this returns 0, because book_id will be different.

0


In [33]:
#looking for duplicates regardless of book_id

duplicate_mask = df.drop(columns=["book_id"]).duplicated(keep=False)
display(df.loc[duplicate_mask].sort_values("title"))

,book_id,title,author,genres,publication_year,page_count,average_rating,rating_count,description
18,B019,The Hobbit,J. R. R. Tolkien,Fantasy;Adventure;Classic,1937,310,4.29,2100000,A reluctant hobbit leaves home with a company ...
60,B061,The Hobbit,J. R. R. Tolkien,Fantasy;Adventure;Classic,1937,310,4.29,2100000,A reluctant hobbit leaves home with a company ...


In [34]:
#This keeps the first matching metadata record. For larger real-world data, we will check editions or ISBNs before merging.

df_clean = df.drop_duplicates(
    subset=df.columns.drop("book_id").tolist()
).copy()
print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))


Original rows: 100
Cleaned rows: 99


In [35]:
#removal verification
print(df_clean.drop(columns=["book_id"]).duplicated().sum())

0


Now let's check data types.
We want to check that publication_year, page_count, average_rating, and rating_count are numeric, and that title, author, genres, and description are text.

In [36]:
df_clean.dtypes

book_id                 str
title                   str
author                  str
genres                  str
publication_year      int64
page_count            int64
average_rating      float64
rating_count          int64
description             str
dtype: object

The data types are all correct, but let's proceed as if they weren't.

In [37]:
numeric_columns = [
    "publication_year", "page_count",
    "average_rating", "rating_count"
]
for column in numeric_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column], errors="coerce"
    )

#Invalid numeric values become missing values (NaN), which we will inspect rather than guess replacements for.

In [38]:
df_clean.isna().sum()

book_id             0
title               0
author              0
genres              0
publication_year    0
page_count          0
average_rating      0
rating_count        0
description         0
dtype: int64

In [39]:
df_clean[df_clean["page_count"].isna()]

,book_id,title,author,genres,publication_year,page_count,average_rating,rating_count,description


Let's look at the formatting of the str type columns

In [40]:
#trims whitespace at start and end of text without affecting missing values

text_columns = ["title", "author", "genres", "description"]

for column in text_columns:
    df_clean[column] = df_clean[column].apply(
        lambda value: value.strip()
        if isinstance(value, str) else value
    )


Now let's standardize the formatting in gnenres

In [41]:

def clean_genres(value):

    #this bit preserves any missing values
    if pd.isna(value):
        return value

    genres = [
        genre.strip()
        for genre in value.split(";") #separates genre at each semicolon
        if genre.strip() #removes white space
    ]

    return ";".join(dict.fromkeys(genres)).apply(clean_genres) 
#join puts the separated, cleaned genres back together 
#dict removes any duplicates
#.apply() applies it to every value in genres column

In [42]:
print(df_clean["genres"].head(10).to_list())

['Science Fiction;Politics;Adventure', 'Science Fiction;Space Opera;Adventure', 'Science Fiction;Politics;Literary', 'Science Fiction;Politics;Literary', 'Science Fiction;Space Opera;Politics', 'Science Fiction;Survival;Adventure', 'Science Fiction;Space;Adventure', 'Science Fiction;Cyberpunk;Crime', 'Science Fiction;Cyberpunk;Adventure', 'Science Fiction;Physics;Mystery']


They're now prepped to be turned into model features in the next stage.

Moving on to the rating values

In [44]:
#checking for any ratings that are negative or greater than 5

invalid_ratings = ~df_clean["average_rating"].between(0, 5)
print("Ratings outside 0–5:", invalid_ratings.sum())

Ratings outside 0–5: 0


In [46]:
#checking for any negative page counts; missing page counts aren't negative, so they're skipped

invalid_pages = df_clean["page_count"].notna() & (df_clean["page_count"] <= 0)
print("Non-positive page counts:", invalid_pages.sum())

Non-positive page counts: 0


In [48]:
print("Invalid years:", ((df_clean["publication_year"] < 0) | (df_clean["publication_year"] > 2026)).sum())

Invalid years: 0


In [50]:
#summary quality checks
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))
print("Duplicate metadata rows:", df_clean.drop(columns=["book_id"]).duplicated().sum())
print("Missing values by column:")
print(df_clean.isna().sum())

Rows: 99
Columns: 9
Duplicate metadata rows: 0
Missing values by column:
book_id             0
title               0
author              0
genres              0
publication_year    0
page_count          0
average_rating      0
rating_count        0
description         0
dtype: int64


Now we can create h

In [ ]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True) #creates the folder if it does not already exist
